# 07 -- Predictor Ranking Table (Fig. 1's companion table)

Reproduces Table 6 of the report (Sec. 5.6): for all 6 model x dataset combinations
(PTQ, loss-based damage), the rank S_raw assigns to the true top-damage layer, and the
size-normalized top-k* overlap of all three scores with the true top-k* damaged layers.

Source CSV: `results/review_response/csv/normalized_ranks_loss.csv`


In [1]:
# Requirements: pandas==3.0.5, numpy==2.5.1, matplotlib==3.11.1, seaborn==0.13.2, scipy==1.18.0
# All notebooks in this report use the same environment; paths below are relative to
# report/notebooks/, so the notebook must be run with its own directory as the working
# directory (the default for `jupyter nbconvert --execute` and for Jupyter's own kernel).
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

REPO = "../.."  # report/notebooks -> report -> repo root
FIG_DIR = "../figures"
import os
os.makedirs(FIG_DIR, exist_ok=True)


In [2]:
RUN = f"{REPO}/results/review_response/csv"
ranks = pd.read_csv(f"{RUN}/normalized_ranks_loss.csv")
ranks


,model,dataset,predictor,n_layers,true_top_layer,true_top_layer_rank,rank_pct,top_k_fixed5_overlap,top_k_norm,top_k_norm_overlap
0,cnn,CIFAR10,raw_trh,6,conv1,2,0.3333,4/5,1,0/1
1,cnn,CIFAR10,dwsq,6,conv1,6,1.0000,4/5,1,0/1
2,cnn,CIFAR10,trh_dwsq,6,conv1,4,0.6667,4/5,1,0/1
3,cnn,IMAGENET100,raw_trh,6,conv1,3,0.5000,4/5,1,0/1
4,cnn,IMAGENET100,dwsq,6,conv1,6,1.0000,4/5,1,0/1
5,cnn,IMAGENET100,trh_dwsq,6,conv1,6,1.0000,4/5,1,0/1
6,resnet18_no_weights,CIFAR10,raw_trh,21,layer1.1.conv1,1,0.0476,3/5,3,1/3
7,resnet18_no_weights,CIFAR10,dwsq,21,layer1.1.conv1,20,0.9524,0/5,3,0/3
8,resnet18_no_weights,CIFAR10,trh_dwsq,21,layer1.1.conv1,12,0.5714,0/5,3,0/3
9,resnet18_no_weights,IMAGENET100,raw_trh,21,conv1,1,0.0476,2/5,3,1/3


Pivot so each (model, dataset) combination becomes one row with all three predictors' top_k_norm_overlap side by side, plus S_raw's rank of the true top-damage layer.


In [3]:
MODEL_LABEL = {"cnn": "CNN", "resnet18_no_weights": "ResNet-18", "resnet50_no_weights": "ResNet-50"}
DATASET_LABEL = {"IMAGENET100": "ImageNet100", "CIFAR10": "CIFAR10"}
N_LAYERS = {"cnn": 6, "resnet18_no_weights": 21, "resnet50_no_weights": 54}
PRED_LABEL = {"raw_trh": "Sraw", "dwsq": "Spert", "trh_dwsq": "Shawq"}

rows = []
for (model, dataset), grp in ranks.groupby(["model", "dataset"]):
    grp = grp.set_index("predictor")
    rows.append({
        "Datensatz": DATASET_LABEL[dataset],
        "Modell (n)": f"{MODEL_LABEL[model]} ({N_LAYERS[model]})",
        "Top-Damage-Layer": grp.loc["raw_trh", "true_top_layer"],
        "Sraw-Rang": int(grp.loc["raw_trh", "true_top_layer_rank"]),
        "Sraw top-k*": grp.loc["raw_trh", "top_k_norm_overlap"],
        "Spert top-k*": grp.loc["dwsq", "top_k_norm_overlap"],
        "Shawq top-k*": grp.loc["trh_dwsq", "top_k_norm_overlap"],
    })

table6 = pd.DataFrame(rows)
# Order rows to match the report: ImageNet100 (CNN, R18, R50) then CIFAR10 (CNN, R18, R50).
model_order = {"CNN": 0, "ResNet-18": 1, "ResNet-50": 2}
table6["_dord"] = table6["Datensatz"].map({"ImageNet100": 0, "CIFAR10": 1})
table6["_mord"] = table6["Modell (n)"].apply(lambda s: model_order[s.split(" (")[0]])
table6 = table6.sort_values(["_dord", "_mord"]).drop(columns=["_dord", "_mord"]).reset_index(drop=True)
table6.to_csv(f"{FIG_DIR}/tab_06_predictor_ranking.csv", index=False)
table6


,Datensatz,Modell (n),Top-Damage-Layer,Sraw-Rang,Sraw top-k*,Spert top-k*,Shawq top-k*
0,ImageNet100,CNN (6),conv1,3,0/1,0/1,0/1
1,ImageNet100,ResNet-18 (21),conv1,1,1/3,0/3,0/3
2,ImageNet100,ResNet-50 (54),conv1,1,1/6,0/6,0/6
3,CIFAR10,CNN (6),conv1,2,0/1,0/1,0/1
4,CIFAR10,ResNet-18 (21),layer1.1.conv1,1,1/3,0/3,0/3
5,CIFAR10,ResNet-50 (54),fc,40,2/6,1/6,1/6


## Output

- `figures/tab_06_predictor_ranking.csv` -- report Table 6
